# VLM teacher-student distillation for ecommerce catalogs

[![Run on Anyscale](https://shields.io/badge/Run_on-Anyscale-009DEF?logo=anyscale)](https://console.anyscale.com/register/ha?utm_source=anyscale_templates&utm_medium=github&utm_campaign=vlm-distillation-catalog-enrichment&redirectTo=/v2/template-preview/vlm-distillation-catalog-enrichment)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/anyscale/templates/tree/main/templates/vlm-distillation-catalog-enrichment)

**Time to complete:** ~45 min

Label a product catalog with a Qwen2.5-VL-7B teacher, distill those labels into a Qwen2.5-VL-3B student via FSDP + LoRA SFT, then run the student in a single Ray Data graph that emits structured catalog JSON *and* SigLIP image+text embeddings. One cluster shape (4× L4), three stages.

## What you'll build

```
Stage 1                    Stage 2                  Stage 3
─────────────              ─────────────            ─────────────────
7B teacher        ───►     3B student SFT  ───►     3B distilled enrich
batch labeling             (FSDP + LoRA)            + SigLIP embeddings
                                                    (one Ray Data graph)

Ray Data LLM               Ray Train + FSDP          Ray Data LLM + Ray Data
   ▼                          ▼                        ▼
teacher.parquet            LoRA adapter             enriched + embedded.parquet
(JSON labels)              (~100 MB)                (JSON + 1152-dim vectors)
```

Every stage runs on the same 4× L4 GPU node (`g6.12xlarge` / `g2-standard-48`). Stages 1 and 2 each take 15–30 min on the smoke configuration; Stage 3 takes ~5 min.

## Set up

This template needs a Hugging Face token (Qwen2.5-VL weights are public but the HF Hub rate-limits anonymous downloads). The token is read from `HF_TOKEN`.

In [ ]:
import os
import subprocess
import sys

# Cache HF downloads on cluster storage so the three stages share weights.
os.environ.setdefault("HF_HOME", "/mnt/cluster_storage/hf_cache")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# Smoke knobs — overridden by tests.sh during CI.
# Set these higher for a production run (N_ROWS=10000 is the default in each script).
N_ROWS = int(os.environ.get("N_ROWS", "20"))
TEACHER_N_ROWS = int(os.environ.get("TEACHER_N_ROWS", str(N_ROWS)))
CATEGORY = os.environ.get("CATEGORY", "Electronics")

os.environ["N_ROWS"] = str(N_ROWS)
os.environ["TEACHER_N_ROWS"] = str(TEACHER_N_ROWS)
os.environ["CATEGORY"] = CATEGORY

print(f"Running with N_ROWS={N_ROWS} TEACHER_N_ROWS={TEACHER_N_ROWS} CATEGORY={CATEGORY}")

In [ ]:
!pip install -q -r requirements.txt

## Stage 1 — Teacher batch labeling

Run Qwen2.5-VL-7B over the catalog and write `{category, attributes, search_tags, description}` JSON per product. The teacher's labels become Stage 2's supervised targets.

Deep-dive notebook: [`notebooks/01_teacher_batch_label.ipynb`](notebooks/01_teacher_batch_label.ipynb)

In [ ]:
!python scripts/run_teacher_batch_label.py

In [ ]:
# Inspect a sample row.
import pyarrow.parquet as pq

teacher_path = f"/mnt/cluster_storage/vlm-distillation-catalog-enrichment/teacher_7b_enriched_{TEACHER_N_ROWS}.parquet"
tbl = pq.read_table(teacher_path)
print(f"rows: {tbl.num_rows}")
print(f"schema: {tbl.schema.names}")
row = tbl.slice(0, 1).to_pylist()[0]
print(f"\nsample title: {row["title"][:100]}")
print(f"sample teacher output (raw_output):\n{row["raw_output"]}")

## Stage 2 — Distill the student with FSDP + LoRA

Fine-tune Qwen2.5-VL-3B on the teacher's parquet using Ray Train. The vision tower stays frozen (standard LLaVA recipe); only the language model gets LoRA adapters. The output is a ~100 MB adapter that drops into Stage 3 as a model swap.

Deep-dive notebook: [`notebooks/02_distill_student_lora.ipynb`](notebooks/02_distill_student_lora.ipynb)

In [ ]:
!python scripts/run_distill_student_lora.py

In [ ]:
# Inspect the adapter.
from pathlib import Path

adapter_dir = Path(f"/mnt/cluster_storage/vlm-distillation-catalog-enrichment/qwen25vl_3b_enrichment_lora_{N_ROWS}")
print(f"adapter dir: {adapter_dir}")
for p in sorted(adapter_dir.rglob("*"))[:10]:
    print(f"  {p.relative_to(adapter_dir)}")

# Point Stage 3 at the freshly trained adapter.
os.environ["QWEN_LORA_ADAPTER_DIR"] = str(adapter_dir)

## Stage 3 — Enrichment + embeddings in one Ray Data graph

Run the LoRA-adapted 3B student to emit catalog JSON, *and* SigLIP-2 (so400m, 1152-dim) image and text embeddings — all in a single streaming Ray Data pipeline. No intermediate disk writes between the VLM and the embedding stages.

Deep-dive notebook: [`notebooks/03_enrich_and_embed.ipynb`](notebooks/03_enrich_and_embed.ipynb)

In [ ]:
!python scripts/run_enrich_and_embed.py

In [ ]:
# Inspect a sample row of the final output.
import pyarrow.parquet as pq

out_path = f"/mnt/cluster_storage/vlm-distillation-catalog-enrichment/enc_vlm_enriched_with_embeddings_{N_ROWS}.parquet"
tbl = pq.read_table(out_path)
print(f"rows: {tbl.num_rows}")
print(f"schema: {tbl.schema.names}")
row = tbl.slice(0, 1).to_pylist()[0]
print(f"\ntitle: {row["title"][:100]}")
print(f"raw_output (3B student): {row["raw_output"]}")
print(f"image_embedding shape: {len(row["image_embedding"])}")
print(f"text_embedding shape: {len(row["text_embedding"])}")

## Production: submit as an Anyscale Job

The same scripts run unchanged as an Anyscale Job. See [`job_config.yaml`](job_config.yaml) for the Stage 3 entry point — the daily/weekly cadence most teams use for catalog refreshes.

```bash
anyscale job submit --config-file job_config.yaml --env HF_TOKEN=$HF_TOKEN
```

Swap the `entrypoint` to `scripts/run_teacher_batch_label.py` or `scripts/run_distill_student_lora.py` to run Stages 1 and 2 as jobs.

## Clean up

In [ ]:
import shutil

shutil.rmtree("/mnt/cluster_storage/vlm-distillation-catalog-enrichment", ignore_errors=True)
print("cluster storage cleared")